# WMAC checks on a second machine: mask verification + position run

Runs **next to** the main run. It reads one file from the main run's Drive folder (`prepared.json`) and writes only to its own folder, `latent-dispatch-wmac-v3-checks`. Code: PR #3 at commit `64044d8`, used as it is; the checks live in one script inside this notebook.

Pick an **A100 (High-RAM)**, then *Run all*. Two things need you in the first minute: allow Google Drive, then the Hugging Face token (only if `"llama"` is in `MODELS`; a Colab secret named `HF_TOKEN` is picked up by itself once you allow this notebook to read it).

**Before you leave:** wait for the line `ready: nothing below waits on you` (about 5 minutes in).

**What it does, per model:**

1. `verify` (about 10 minutes): on 48 layouts (the 24 diagnostic ones + 24 unselected ones),
   - swap A's text for other text of the same length: under the blocked mask B's keys and values must not change at any layer; under the full mask they must;
   - layer by layer, how far apart the pairs that should be identical are (joint vs full mask, separately encoded blocks vs blocked mask), with the default attention kernels and with one kernel for both sides.
2. `position` (about 2 hours): the same 200 sources with A's supporting paragraph moved to the **front** of A. Conditions: blocked, gold, full, joint, and `matched_late` = as many A tokens as the support has, taken from the **end** of A, where the support used to be. 2,000 answers.

**Time and compute units** (estimates from the main run's speed on 11-17: 3.7 s per answer on Qwen3-8B, 4.0 s on Llama; A100 High-RAM at 6.73 units per hour): Qwen3-8B about 2.4 h and 16 units; Llama about 2.5 h and 17 units; both about 5 h and 33 units. To run only Qwen3-8B, delete `"llama"` from `MODELS`. To drop the joint condition (saves about 25 minutes per model), delete `,b1` from `CONDITIONS`.

**What goes to Drive** (folder `latent-dispatch-wmac-v3-checks`), every 5 minutes and when each part ends: `<model>/verify/rows.jsonl` + `summary.json`, `<model>/position/rows.jsonl` + `plan.json` + `status.json`, and `log.txt`.

**If the session dies:** open this notebook on a new A100 and *Run all*. It pulls back what is in Drive and continues; answers already saved are not redone.


In [ ]:
REPO, COMMIT = "https://github.com/ajit-mallav/latent-dispatch.git", "64044d8c8db029af3cbede5984f02bf9ff43740e"
ROOT = "/content/latent-dispatch-wmac-v3"                            # PR #3's code at that commit
MAIN_RUN = "/content/drive/MyDrive/latent-dispatch-wmac-v3"          # the main run's Drive folder; only prepared.json is read
BACKUP = "/content/drive/MyDrive/latent-dispatch-wmac-v3-checks"     # everything this notebook saves
OUT = "/content/wmac-checks"                                         # results on this machine
SCRIPT = "/content/wmac_checks.py"
PREPARED = "/content/prepared.json"
MODELS = ["8b", "llama"]                 # delete "llama" to run only Qwen3-8B
PARTS = ["verify", "position"]
CONDITIONS = "blocked,matched_late,gold,full,b1"
SNAPSHOT_MINUTES = 5
LLAMA = "meta-llama/Llama-3.1-8B-Instruct"


In [ ]:
# The two steps that need you: Google Drive, then (for Llama) the Hugging Face token.
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(BACKUP, exist_ok=True)
assert os.path.exists(f"{MAIN_RUN}/prepared.json"), "prepared.json is not in the main run's Drive folder"

if "llama" in MODELS:
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face Read token: ").strip()
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import hf_hub_download
    hf_hub_download(LLAMA, "config.json")   # fails here, not hours in, if this account is not approved for Llama
    print("Llama access OK")


In [ ]:
import collections, hashlib, json, os, shutil, subprocess, threading, time

os.environ["TZ"] = "America/New_York"
time.tzset()


def sh(cmd):
    """Run a command, show its output as it comes, return its exit code."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()


def must(cmd):
    if sh(cmd):
        raise RuntimeError(f"failed: {cmd}")


def note(message):
    """Print one line and add it to log.txt in Drive."""
    line = time.strftime("%m-%d %H:%M:%S ") + message
    print(line, flush=True)
    try:
        with open(f"{BACKUP}/log.txt", "a") as f:
            f.write(line + "\n")
    except Exception as error:
        print("  (log.txt not written:", type(error).__name__, error, ")", flush=True)


def put(source, name):
    """Copy one file into the Drive folder. A half-written copy never replaces a good one."""
    target, partial = f"{BACKUP}/{name}", f"{BACKUP}/{name}.uploading"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    shutil.copyfile(source, partial)
    try:
        os.replace(partial, target)
    except OSError:
        shutil.copyfile(source, target)
        os.remove(partial)


def results(folder):
    """The small result files under a folder, as paths relative to it."""
    found = []
    for base, _, names in os.walk(folder):
        for name in sorted(names):
            if name.endswith((".jsonl", ".json")) and name != "log.txt":
                found.append(os.path.relpath(os.path.join(base, name), folder))
    return sorted(found)


def backup():
    """Copy every result file to Drive. Safe while a part is running: a row cut in half is redone on restart."""
    for name in results(OUT):
        put(f"{OUT}/{name}", name)
    return len(results(OUT))


def restore():
    """On a new machine: bring back what Drive has, so saved answers are not redone."""
    if results(OUT):
        return 0
    names = results(BACKUP)
    for name in names:
        os.makedirs(os.path.dirname(f"{OUT}/{name}"), exist_ok=True)
        shutil.copyfile(f"{BACKUP}/{name}", f"{OUT}/{name}")
    return len(names)


def saved(model, part):
    path = f"{OUT}/{model}/{part}/rows.jsonl"
    if not os.path.exists(path):
        return 0
    with open(path, "rb") as f:
        return sum(1 for line in f if line.endswith(b"\n"))


def watch(stop):
    while not stop.wait(SNAPSHOT_MINUTES * 60):
        try:
            backup()
        except Exception as error:
            print(f"[backup] failed, will retry: {type(error).__name__}: {error}", flush=True)


In [ ]:
import torch

assert torch.cuda.is_available(), "no GPU: pick an A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), f"{gb:.0f} GB")
assert gb > 30, "pick an A100: the 8B models in bfloat16 with these contexts need more than 30 GB"
if not os.path.exists(ROOT):
    must(f"rm -rf {ROOT}.tmp && git clone -q {REPO} {ROOT}.tmp && git -C {ROOT}.tmp checkout -q {COMMIT} && mv {ROOT}.tmp {ROOT}")
os.chdir(ROOT)
must("git log -1 --format='running commit %h (%s)'")
must("pip install -q -e '.[dev]'")


In [ ]:
%%writefile /content/wmac_checks.py
#!/usr/bin/env python
"""Two follow-up checks for the WMAC paper. Uses the PR #3 library as it is (commit 64044d8); changes nothing in it.

verify     Does the attention mask do only what it says? On real weights, for each layout:
           swap   Replace A's tokens with other text of the same length.
                  Under the blocked mask, B's keys and values must not change at any layer (B cannot see A).
                  Under the full mask they must change (this shows the test is able to fail).
           pairs  Layer by layer, how far apart are the two computations that should be the same?
                    joint reading            vs  the mask code with nothing blocked
                    separately encoded blocks vs  the mask code with A blocked   (both at true positions)
                  Measured twice: with PyTorch's default attention kernels (the mask code and the plain code use
                  different ones), then with one kernel for both sides.

position   The same MuSiQue layouts with A's supporting paragraph moved from the end of A to the front, so it is
           far from B. Conditions (thinking off, same decoding as the main run):
             blocked, gold, full, b1 (joint), and
             matched_late: B sees as many A tokens as the support has, taken from the END of A's non-support text,
                           the place the support used to be. Position now favours the control.

Rows are appended to <out>/<model>/<part>/rows.jsonl as they are produced; a rerun skips rows already saved.

  python wmac_checks.py selftest                                     # seconds, CPU, tiny random models
  python wmac_checks.py preflight --prepared prepared.json           # tokenizers only: is every reordered layout usable?
  python wmac_checks.py run --prepared prepared.json --model 8b --out results [--parts verify,position]
"""
import argparse
import contextlib
import copy
import hashlib
import json
import os
import statistics
import subprocess
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import torch

from latent_dispatch import arms, kv, masking, paper_experiments, prompts, wmac_experiments
from latent_dispatch.model import Model
from latent_dispatch.paper_plan import digest
from latent_dispatch.score import contains_answer

PREPARED_SHA256 = "8abf29425611b1f3b65f5016fd40651ef0281bdb6de9852e1df2940f33953058"   # the main run's question set
CONDITIONS = ("blocked", "matched_late", "gold", "full", "b1")
LAYOUT = "support-first-v1"
UNSELECTED = 24            # verify: this many layouts of the main run, evenly spaced, next to the 24 diagnostic ones


# ------------------------------------------------------------------------------------------------ small helpers

class Rows:
    """Append-only JSONL, one row per key. A row cut off when the machine stopped is ignored and redone."""

    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.rows = {}
        if self.path.exists():
            raw = self.path.read_bytes()
            for line in raw.split(b"\n"):
                if line.strip():
                    try:
                        row = json.loads(line)
                    except ValueError:
                        continue
                    self.rows[row["key"]] = row
            if raw and not raw.endswith(b"\n"):
                with self.path.open("ab") as f:
                    f.write(b"\n")

    def add(self, key, row):
        row = {"key": key, **row}
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
        self.rows[key] = row


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=1, ensure_ascii=False) + "\n")
    tmp.replace(path)


def frozen(path, value):
    """Saved rows may only be continued under the same plan."""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != json.loads(json.dumps(value)):
            raise ValueError(f"{path}: the plan differs from the one these rows were made under; use a new --out")
    else:
        write_json(path, value)


def load_bundle(path, check_sha=True):
    raw = Path(path).read_bytes()
    sha = hashlib.sha256(raw).hexdigest()
    if check_sha and sha != PREPARED_SHA256:
        raise ValueError(f"{path} is not the main run's question set (sha256 {sha[:12]}..., expected {PREPARED_SHA256[:12]}...)")
    return json.loads(raw)


def load_model(bundle, key, device=None, dtype="bfloat16", repo=None, fmt=None):
    """The model exactly as the main run loaded it. repo/fmt replace it with a small model for a local smoke test."""
    cfg, entry = bundle["config"], bundle["config"]["models"][key]
    torch.manual_seed(cfg["seed"])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    if repo:
        return Model.load(repo, device=device, dtype=dtype, fmt=fmt, attn_implementation=cfg["attention"])
    return Model.load(entry["name"], device=device, dtype=dtype, fmt=entry["format"], revision=entry["revision"],
                      attn_implementation=cfg["attention"])


def runtime(model):
    import transformers
    cuda = model.device.type == "cuda"
    return dict(model=model.name, revision=getattr(model.hf.config, "_commit_hash", None), dtype=model.dtype_name,
                device=str(model.device), device_name=torch.cuda.get_device_name(0) if cuda else model.device.type,
                torch=torch.__version__, transformers=transformers.__version__,
                attention=getattr(model.hf.config, "_attn_implementation", None))


def code_commit():
    try:
        root = Path(masking.__file__).resolve().parents[1]
        return subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def free(model):
    if model.device.type == "cuda":
        torch.cuda.empty_cache()


# ------------------------------------------------------------------------------------------------------ verify

def layer_gap(a, b, rows=slice(None)):
    """Per layer, the largest absolute difference between two caches' keys and values over the given positions,
    and that difference as a share of the largest key or value in the first cache."""
    keys, values, identical, relative = [], [], [], []
    for (ka, va), (kb, vb) in zip(kv.layers(a), kv.layers(b)):
        ka, kb, va, vb = ka[..., rows, :].float(), kb[..., rows, :].float(), va[..., rows, :].float(), vb[..., rows, :].float()
        keys.append((ka - kb).abs().max().item())
        values.append((va - vb).abs().max().item())
        identical.append(bool(torch.equal(ka, kb) and torch.equal(va, vb)))
        relative.append(max(keys[-1] / ka.abs().max().clamp_min(1e-12).item(), values[-1] / va.abs().max().clamp_min(1e-12).item()))
    return dict(layers=len(keys), identical_layers=sum(identical), largest=max(keys + values), largest_relative=max(relative),
                keys_by_layer=[float(f"{x:.4g}") for x in keys], values_by_layer=[float(f"{x:.4g}") for x in values],
                relative_by_layer=[float(f"{x:.3g}") for x in relative])


def first_logits(model, cache, pos):
    """Scores for the first answer token (the answer prompt is fed on a copy, so the cache is not changed)."""
    tail = model.ids(prompts.tail(model.fmt, "off"))
    logits = model.forward(tail, kv.copy(cache), pos).logits[0, -1].float()
    if not torch.isfinite(logits).all():
        raise ValueError("non-finite logits")
    return logits


def logit_gap(x, y):
    top = torch.topk(x, 2).values
    return dict(largest=(x - y).abs().max().item(), same_top_token=bool(x.argmax() == y.argmax()),
                top2_margin=(top[0] - top[1]).item())


def swapped_ids(model, q, donor):
    """q's token ids with A's tokens replaced by the donor question's A tokens, cut or repeated to the same length."""
    ids, sizes, n_prefix = paper_experiments.context_tokens(model, q)
    other = model.ids(prompts.block(donor["workers"][0]))
    filler = [other[i % len(other)] for i in range(sizes[0])]
    changed = sum(x != y for x, y in zip(filler, ids[n_prefix:n_prefix + sizes[0]])) / sizes[0]
    return ids, ids[:n_prefix] + filler + ids[n_prefix + sizes[0]:], n_prefix + sizes[0], changed


def swap_check(model, q, donor):
    """Change A's content, keep everything else. -> how much B's keys, values and last logits moved, per mask."""
    ids, new, b_start, changed = swapped_ids(model, q, donor)
    result = {"a_tokens_changed": round(changed, 4)}
    for condition in ("blocked", "full"):
        layout_ids, mask, _, _ = masking.layout(model, q, condition)
        if layout_ids != ids or len(new) != len(ids):
            raise ValueError("the swap changed the token sequence outside A")
        before = model.forward(ids, attention_mask=mask)
        after = model.forward(new, attention_mask=mask)
        gap = layer_gap(before.past_key_values, after.past_key_values, slice(b_start, None))
        x, y = before.logits[0, -1].float(), after.logits[0, -1].float()
        result[condition] = {**gap, "last_logits_largest": (x - y).abs().max().item(),
                             "last_logits_identical": bool(torch.equal(before.logits[0, -1], after.logits[0, -1]))}
        del before, after, mask
        free(model)
    return result


def pair_check(model, q):
    """The two pairs that should be one computation. -> (differences, the joint cache, the blocked-mask cache)"""
    joint, pos, _ = arms.b1_context(model, q)
    full, _, _, _ = masking.context(model, q, "full")
    separate, _, _ = arms.k0_context(model, q, positions="true")
    blocked, _, _, _ = masking.context(model, q, "blocked")
    result = {}
    for name, reference, candidate in (("joint_vs_full_mask", joint, full), ("separate_vs_blocked_mask", separate, blocked)):
        result[name] = {**layer_gap(reference, candidate),
                        "first_logits": logit_gap(first_logits(model, reference, pos), first_logits(model, candidate, pos))}
    del full, separate
    free(model)
    return result, joint, blocked


@contextlib.contextmanager
def one_kernel(model):
    """Make every attention call use the same PyTorch kernel (the plain 'math' one). Only CUDA has several."""
    if model.device.type != "cuda":
        yield False
        return
    from torch.nn.attention import SDPBackend, sdpa_kernel
    with sdpa_kernel([SDPBackend.MATH]):
        yield True


def verify_layout(model, q, donor):
    tokens, sizes, _ = paper_experiments.context_tokens(model, q)
    row = dict(id=q["id"], context_tokens=len(tokens), worker_tokens=sizes, swap=swap_check(model, q, donor))
    default, joint, blocked = pair_check(model, q)
    row["pairs_default_kernels"] = default
    row["pairs_one_kernel"] = row["kernel_change_alone"] = None
    with one_kernel(model) as active:
        if active:
            try:
                same, joint_one, blocked_one = pair_check(model, q)
                row["pairs_one_kernel"] = same
                row["kernel_change_alone"] = dict(joint=layer_gap(joint, joint_one), blocked_mask=layer_gap(blocked, blocked_one))
                del joint_one, blocked_one
            except RuntimeError as error:     # e.g. out of memory: the plain kernel builds the whole attention table
                row["one_kernel_error"] = f"{type(error).__name__}: {error}"[:400]
    del joint, blocked
    free(model)
    return row


def verify_ids(bundle, limit=None):
    """The 24 diagnostic layouts (picked earlier because results disagreed or agreed) + 24 unselected ones."""
    main = bundle["musique_ids"]
    step = max(1, len(main) // UNSELECTED) | 1          # odd, so the picks alternate between the AB and BA layouts
    groups = [("diagnostic", i) for i in bundle["validation_ids"]] + [("unselected", i) for i in main[::step][:UNSELECTED]]
    if limit:
        groups = groups[:limit // 2] + [g for g in groups if g[0] == "unselected"][:limit - limit // 2]
    return groups


def verify_summary(rows):
    def med_max(values):
        values = list(values)
        return dict(median=statistics.median(values), largest=max(values)) if values else None

    out = dict(layouts=len(rows))
    blocked = [r["swap"]["blocked"] for r in rows]
    full = [r["swap"]["full"] for r in rows]
    out["swap_blocked"] = dict(
        b_identical_at_every_layer=sum(b["identical_layers"] == b["layers"] and b["last_logits_identical"] for b in blocked),
        largest_difference_anywhere=max(max(b["largest"], b["last_logits_largest"]) for b in blocked))
    out["swap_full"] = dict(b_changed=sum(f["largest"] > 0 for f in full), smallest_of_the_largest_differences=min(f["largest"] for f in full))
    for name in ("pairs_default_kernels", "pairs_one_kernel"):
        have = [r[name] for r in rows if r.get(name)]
        if not have:
            out[name] = None
            continue
        out[name] = {}
        for pair in ("joint_vs_full_mask", "separate_vs_blocked_mask"):
            items = [h[pair] for h in have]
            out[name][pair] = dict(
                layouts=len(items), identical_at_every_layer=sum(i["identical_layers"] == i["layers"] for i in items),
                first_layer_keys=med_max(i["keys_by_layer"][0] for i in items),
                last_layer_keys=med_max(i["keys_by_layer"][-1] for i in items),
                last_layer_values=med_max(i["values_by_layer"][-1] for i in items),
                first_logits=med_max(i["first_logits"]["largest"] for i in items),
                same_top_token=sum(i["first_logits"]["same_top_token"] for i in items),
                top_token_flips_with_margin_under_1=sum((not i["first_logits"]["same_top_token"]) and i["first_logits"]["top2_margin"] < 1
                                                        for i in items))
    have = [r["kernel_change_alone"] for r in rows if r.get("kernel_change_alone")]
    out["kernel_change_alone"] = {side: med_max(h[side]["largest"] for h in have) for side in ("joint", "blocked_mask")} if have else None
    return out


def run_verify(model, bundle, folder, limit=None):
    questions = bundle["questions"]
    groups = verify_ids(bundle, limit)
    rows = Rows(folder / "rows.jsonl")
    start = time.monotonic()
    for n, (group, id) in enumerate(groups):
        if id in rows.rows:
            continue
        donor = questions[groups[(n + 1) % len(groups)][1]]
        if donor["source_id"] == questions[id]["source_id"]:
            donor = questions[groups[(n + 2) % len(groups)][1]]
        t = time.monotonic()
        row = verify_layout(model, questions[id], donor)
        rows.add(id, {"group": group, **row, "seconds": time.monotonic() - t})
        b = row["swap"]["blocked"]
        print(f"  verify {n + 1}/{len(groups)} {group:10s} {row['context_tokens']:5d} tokens | swap under blocked: "
              f"{b['identical_layers']}/{b['layers']} layers identical, largest diff {b['largest']:.3g} | under full: "
              f"largest diff {row['swap']['full']['largest']:.3g} | {time.monotonic() - start:.0f} s", flush=True)
    summary = dict(runtime=runtime(model), commit=code_commit(), all=verify_summary(list(rows.rows.values())),
                   **{g: verify_summary([r for r in rows.rows.values() if r["group"] == g])
                      for g in ("diagnostic", "unselected") if any(r["group"] == g for r in rows.rows.values())})
    write_json(folder / "summary.json", summary)
    s = summary["all"]
    print(f"verify, {s['layouts']} layouts:")
    print(f"  swap under the blocked mask: B identical at every layer in {s['swap_blocked']['b_identical_at_every_layer']} of {s['layouts']}"
          f" (largest difference anywhere {s['swap_blocked']['largest_difference_anywhere']:.3g})")
    print(f"  swap under the full mask:    B changed in {s['swap_full']['b_changed']} of {s['layouts']}")
    for name in ("pairs_default_kernels", "pairs_one_kernel"):
        if s[name]:
            for pair, v in s[name].items():
                print(f"  {name}, {pair}: first-token logits differ by median {v['first_logits']['median']:.3g}, "
                      f"largest {v['first_logits']['largest']:.3g}; same top token in {v['same_top_token']} of {v['layouts']}; "
                      f"identical at every layer in {v['identical_at_every_layer']}")
    failed = sum("one_kernel_error" in r for r in rows.rows.values())
    if failed:
        print(f"  the one-kernel pass failed on {failed} layouts (see one_kernel_error in rows.jsonl); the rest is unaffected")
    if s["kernel_change_alone"]:
        print("  kernel change alone (same computation, two kernels): largest key/value difference "
              f"joint {s['kernel_change_alone']['joint']['largest']:.3g}, blocked mask {s['kernel_change_alone']['blocked_mask']['largest']:.3g}")
    return summary


# ---------------------------------------------------------------------------------------------------- position

def support_first(q):
    """A copy of the layout with A's supporting paragraph moved to the front of A. B is not touched."""
    q = copy.deepcopy(q)
    refs = [g for g in q["gold"] if g["worker"] == 0]
    if len(refs) != 1:
        raise ValueError(f"{q['id']}: need exactly one supporting paragraph in A")
    index, paragraphs = refs[0]["index"], q["workers"][0]
    q["workers"][0] = [paragraphs[index]] + [p for i, p in enumerate(paragraphs) if i != index]
    refs[0]["index"] = 0
    q["layout_protocol"] = LAYOUT
    return q


def late_window(model, q):
    """A-token indices of the late control: as many tokens as the support has, from the END of A's later non-support
    text, skipping paragraphs that name the answer or the bridge. The mirror of wmac_experiments.matched_indices."""
    _, spans = masking.paragraph_tokens(model, q["workers"][0])
    gold = sorted(g["index"] for g in q["gold"] if g["worker"] == 0)
    if len(gold) != 1:
        raise ValueError("the late control needs exactly one supporting paragraph in A")
    n = len(spans[gold[0]])
    names = q["answers"] + q.get("bridges", [])
    candidates = [i for i in range(gold[0] + 1, len(spans)) if not contains_answer(q["workers"][0][i], names)]
    eligible = [t for i in candidates for t in spans[i]]
    if len(eligible) < n:
        raise ValueError(f"{q['id']}: not enough later non-support tokens ({len(eligible)} < {n})")
    chosen = eligible[-n:]
    picked = set(chosen)
    a_tokens = len(model.ids(prompts.block(q["workers"][0])))
    return chosen, dict(a_visible_tokens=n, a_gold_tokens=n, a_paragraphs=[i for i in candidates if picked & set(spans[i])],
                        a_visible_indices=chosen, a_gold_indices=spans[gold[0]],
                        tokens_between_window_and_b=a_tokens - 1 - chosen[-1],
                        tokens_between_gold_and_b=a_tokens - 1 - spans[gold[0]][-1], window_follows_gold=True,
                        control="equal-token later non-support window nearest B; its keys were computed after the support")


def late_layout(model, q):
    """Token ids and attention mask for the late control: B may look at the chosen A tokens and nothing else in A."""
    selected, extra = late_window(model, q)
    ids, mask, sizes, meta = masking.layout(model, q, "blocked")
    n_prefix = len(ids) - sum(sizes)
    mask[0, 0, n_prefix + sizes[0]:, [n_prefix + i for i in selected]] = 0
    return ids, mask, sizes, {**meta, **extra, "condition": "matched_late"}


def generate_late(model, q, answer_tokens):
    """wmac_experiments.generate for the late control (thinking off)."""
    ids, mask, sizes, meta = late_layout(model, q)
    out = model.forward(ids, attention_mask=mask)
    cache, pos = out.past_key_values, len(ids)
    del out, mask
    expected, expected_sizes, _ = paper_experiments.context_tokens(model, q)
    if ids != expected or sizes != expected_sizes or kv.length(cache) != pos:
        raise ValueError("the late control changed the token sequence or the cache length")
    gen = model.greedy(model.ids(prompts.tail(model.fmt, "off")), cache, pos, max_new=answer_tokens)
    return {**paper_experiments.generation_fields(gen), "worker_tokens": sizes, "mask": meta,
            "context_tokens_sha256": digest(expected), "final_answer_position": pos}


def position_layouts(model, questions, ids, max_tokens):
    """Reorder every layout and check it before any answer is generated. A source is kept only if both of its
    layouts pass. -> (kept {id: reordered question}, dropped {id: reason})"""
    kept, dropped = {}, {}
    for id in ids:
        try:
            q = support_first(questions[id])
            tokens, _, _ = paper_experiments.context_tokens(model, q)
            pieces = [prompts.prefix(model.fmt, q["question"]), *[prompts.block(w) for w in q["workers"]],
                      prompts.tail(model.fmt, "off")]
            if [t for piece in pieces for t in model.ids(piece)] != model.ids("".join(pieces)):
                raise ValueError("tokenizing the pieces differs from tokenizing the whole prompt")
            if len(tokens) > max_tokens:
                raise ValueError(f"context of {len(tokens)} tokens is over {max_tokens}")
            late_window(model, q)
            kept[id] = q
        except ValueError as error:
            dropped[id] = str(error)
    by_source = {}
    for id in ids:
        by_source.setdefault(questions[id]["source_id"], []).append(id)
    for source, members in by_source.items():
        if any(m in dropped for m in members):
            for m in members:
                if m in kept:
                    del kept[m]
                    dropped[m] = "the other layout of this source was dropped"
    return kept, dropped


def position_ids(bundle, sources=None):
    questions, ids = bundle["questions"], bundle["musique_ids"]
    order = list(dict.fromkeys(questions[i]["source_id"] for i in ids))
    keep = set(order[:sources] if sources else order)
    return [i for i in ids if questions[i]["source_id"] in keep]


def run_position(model, bundle, folder, conditions=CONDITIONS, sources=None):
    cfg, questions = bundle["config"], bundle["questions"]
    kept, dropped = position_layouts(model, questions, position_ids(bundle, sources), cfg["max_context_tokens"])
    info, commit = runtime(model), code_commit()
    frozen(folder / "plan.json", dict(layout_protocol=LAYOUT, conditions=list(conditions), model=info["model"], revision=info["revision"],
                                      dtype=info["dtype"], commit=commit, answer_tokens=cfg["answer_tokens"],
                                      question_sha256={id: digest(q) for id, q in kept.items()}, dropped=dropped))
    print(f"position: {len(kept)} layouts ({len(kept) // 2} sources) x {len(conditions)} conditions = {len(kept) * len(conditions)} answers; "
          f"dropped before any answer: {len(dropped)}", flush=True)
    rows = Rows(folder / "rows.jsonl")
    todo = [(id, c) for id in kept for c in conditions if f"{id}|{c}" not in rows.rows]
    start, done = time.monotonic(), 0
    for id, condition in todo:
        q, t = kept[id], time.monotonic()
        if condition == "matched_late":
            fields = generate_late(model, q, cfg["answer_tokens"])
        else:
            fields = wmac_experiments.generate(model, q, condition, "off", cfg["think_budget"], cfg["answer_tokens"])
        rows.add(f"{id}|{condition}", dict(
            id=id, condition=f"{condition}_off", layout_protocol=LAYOUT, commit=commit, question_sha256=digest(q),
            options=dict(condition=condition, thinking="off", answer_tokens=cfg["answer_tokens"], think_budget=cfg["think_budget"]),
            device_name=info["device_name"], seconds=time.monotonic() - t, **fields))
        done += 1
        if done % 25 == 0 or done == len(todo):
            pace = (time.monotonic() - start) / done
            print(f"  position {len(rows.rows)}/{len(kept) * len(conditions)} answers, {pace:.2f} s each, "
                  f"about {(len(todo) - done) * pace / 60:.0f} min left", flush=True)
        free(model)
    complete = len(rows.rows) == len(kept) * len(conditions)
    write_json(folder / "status.json", dict(complete=complete, saved_rows=len(rows.rows), expected_rows=len(kept) * len(conditions),
                                            layouts=len(kept), dropped=len(dropped), runtime=info))
    return complete


# ------------------------------------------------------------------------------------------ tokenizer preflight

def tokenizer_only(name, fmt, revision=None):
    """Enough of a Model for the layout checks: no weights."""
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(name, **({"revision": revision} if revision else {}))
    return SimpleNamespace(tok=tok, fmt=prompts.FORMATS[fmt], ids=lambda text: tok(text, add_special_tokens=False).input_ids)


def run_preflight(bundle, key, repo=None, fmt=None):
    entry = bundle["config"]["models"][key]
    stand_in = tokenizer_only(repo or entry["name"], fmt or entry["format"], None if repo else entry["revision"])
    want = bundle.get("tokenizer_sha256", {}).get(key)
    same = digest(stand_in.tok.backend_tokenizer.to_str()) == want
    ids = position_ids(bundle)
    kept, dropped = position_layouts(stand_in, bundle["questions"], ids, bundle["config"]["max_context_tokens"])
    gaps = sorted(late_window(stand_in, q)[1]["tokens_between_window_and_b"] for q in kept.values())
    far = sorted(late_window(stand_in, q)[1]["tokens_between_gold_and_b"] for q in kept.values())
    print(f"{key}: tokenizer {'matches' if same else 'DOES NOT MATCH'} the main run's; {len(kept)} of {len(ids)} layouts usable, "
          f"{len(dropped)} dropped {sorted(set(dropped.values()))[:3]}")
    if gaps:
        print(f"   control window ends {gaps[0]}..{gaps[-1]} tokens before B (median {gaps[len(gaps) // 2]}; within 3 tokens in "
              f"{sum(g <= 3 for g in gaps)}); support ends {far[0]}..{far[-1]} tokens before B (median {far[len(far) // 2]})")
    return same, kept, dropped


# ---------------------------------------------------------------------------------------------------- selftest

def selftest():
    """The script's own logic on tiny random models (CPU, float32). Run from the repository folder."""
    sys.path.insert(0, os.getcwd())
    import tempfile
    from tests.conftest import make_question, tiny_model
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
    q = make_question(
        [["Aria Venn was born in Dunmere, a quiet town by the sea.", "The harbor of Dunmere closes every winter for repairs.",
          "Velmora is known for its long covered market.", "People: Ines Farrow's favorite city is Velmora.",
          "Old maps show three bridges across the Calder river."],
         ["The river the Calder flows through Irvale.", "Cities: The mayor of Velmora is Bram Holt."]],
        gold=((0, 0, 3), (1, 1, 1)), id="t:AB")
    donor = make_question([["Tomas Reed keeps bees near the northern hills.", "The hills are covered in heather each August."],
                           ["Cities: The mayor of Irvale is Odile Marsh."]], gold=((0, 0, 0), (1, 1, 0)), id="d:AB")
    donor["source_id"] = "d"
    for family in ("qwen3", "llama3"):
        m = tiny_model(family, tok)
        # 1. swap: B cannot move under blocked, must move under full
        s = swap_check(m, q, donor)
        assert s["blocked"]["identical_layers"] == s["blocked"]["layers"] and s["blocked"]["last_logits_identical"], s["blocked"]
        assert s["blocked"]["largest"] == 0 and s["full"]["largest"] > 0 and s["a_tokens_changed"] > 0.5, s
        # 2. the pairs agree to float32 rounding on a CPU, where there is one kernel
        pairs, joint, blocked = pair_check(m, q)
        assert all(p["largest"] < 1e-4 and p["first_logits"]["largest"] < 1e-3 for p in pairs.values()), pairs
        with one_kernel(m) as active:
            assert active is False
        # 3. support first: only the support moves; the original is untouched
        moved = support_first(q)
        assert moved["workers"][0][0] == q["workers"][0][3] and moved["workers"][0][1:] == [p for i, p in enumerate(q["workers"][0]) if i != 3]
        assert [g["index"] for g in moved["gold"] if g["worker"] == 0] == [0] and q["gold"][0]["index"] == 3
        assert moved["workers"][1] == q["workers"][1] and moved["layout_protocol"] == LAYOUT
        # 4. late window: support-sized, after the support, at the end of the eligible text, not in the bridge paragraph
        chosen, meta = late_window(m, moved)
        _, spans = masking.paragraph_tokens(m, moved["workers"][0])
        bridge_paragraph = next(i for i, p in enumerate(moved["workers"][0]) if "Velmora" in p and i != 0)
        assert len(chosen) == len(spans[0]) == meta["a_visible_tokens"] and min(chosen) > max(spans[0])
        assert not set(chosen) & set(spans[bridge_paragraph]) and chosen[-1] == spans[len(spans) - 1][-1]
        assert chosen == sorted(chosen) and meta["tokens_between_gold_and_b"] > meta["tokens_between_window_and_b"]
        # 5. the late mask opens exactly those tokens to B, and nothing else differs from the blocked mask
        ids, mask, sizes, meta2 = late_layout(m, moved)
        b_ids, b_mask, _, _ = masking.layout(m, moved, "blocked")
        n_prefix = len(ids) - sum(sizes)
        b_start = n_prefix + sizes[0]
        assert ids == b_ids and meta2["condition"] == "matched_late"
        assert (mask[0, 0, b_start, n_prefix:b_start] == 0).nonzero().flatten().tolist() == chosen
        opened = (mask != b_mask).nonzero()
        assert set(opened[:, 2].tolist()) == set(range(b_start, len(ids))) and set((opened[:, 3] - n_prefix).tolist()) == set(chosen)
        # 6. the late control really generates, and the standard conditions run on the reordered layout
        row = generate_late(m, moved, 4)
        assert row["final_answer_position"] == len(ids) and row["mask"]["a_visible_indices"] == chosen
        for condition in ("blocked", "gold", "full", "b1"):
            assert wmac_experiments.generate(m, moved, condition, "off", 8, 4)["final_answer_position"] == len(ids)
        # 7. layout check drops a source whose other layout fails
        bad = copy.deepcopy(q)
        bad["id"], bad["workers"][0] = "t:BA", [q["workers"][0][3]]              # no later text for the control
        bad["gold"][0]["index"] = 0
        kept, dropped = position_layouts(m, {"t:AB": q, "t:BA": bad}, ["t:AB", "t:BA"], 4096)
        assert kept == {} and "not enough later" in dropped["t:BA"] and "other layout" in dropped["t:AB"]
    if torch.cuda.is_available():            # the same checks through the GPU code path
        gpu = Model(tiny_model("qwen3", tok).hf.to("cuda"), tok, "tiny-qwen3-cuda", fmt="qwen3")
        row = verify_layout(gpu, q, donor)
        assert row["swap"]["blocked"]["largest"] == 0 and row["swap"]["full"]["largest"] > 0, row["swap"]
        if row["pairs_one_kernel"] is None:
            print("selftest WARNING: the one-kernel comparison does not run here; the run will record that and carry on:",
                  row.get("one_kernel_error"))
        else:
            print("selftest: GPU path ok; kernel change alone:", {k: v["largest"] for k, v in row["kernel_change_alone"].items()})
    # 8. rows survive a cut-off line
    with tempfile.TemporaryDirectory() as folder:
        rows = Rows(Path(folder) / "rows.jsonl")
        rows.add("a", {"x": 1})
        with rows.path.open("a") as f:
            f.write('{"key": "b", "x"')                                           # the machine stopped here
        again = Rows(rows.path)
        assert list(again.rows) == ["a"]
        again.add("b", {"x": 2})
        assert list(Rows(rows.path).rows) == ["a", "b"]
    print("selftest passed")


# -------------------------------------------------------------------------------------------------------- main

def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("command", choices=["selftest", "preflight", "run"])
    p.add_argument("--prepared")
    p.add_argument("--model", default="8b", help="8b or llama: the main run's entry for that model")
    p.add_argument("--out", default="results")
    p.add_argument("--parts", default="verify,position")
    p.add_argument("--conditions", default=",".join(CONDITIONS))
    p.add_argument("--sources", type=int, help="position: only the first N sources (default all)")
    p.add_argument("--limit", type=int, help="verify: only N layouts (default 48)")
    p.add_argument("--device")
    p.add_argument("--dtype", default="bfloat16")
    p.add_argument("--repo", help="small local smoke test: another model in place of the main run's")
    p.add_argument("--fmt", help="chat format for --repo")
    args = p.parse_args()
    if args.command == "selftest":
        return selftest()
    bundle = load_bundle(args.prepared)
    if args.command == "preflight":
        for key in args.model.split(","):
            run_preflight(bundle, key, args.repo, args.fmt)
        return None
    conditions = tuple(args.conditions.split(","))
    if set(conditions) - set(CONDITIONS):
        raise ValueError(f"conditions must be among {CONDITIONS}")
    model = load_model(bundle, args.model, args.device, args.dtype, args.repo, args.fmt)
    print("loaded", runtime(model), flush=True)
    folder = Path(args.out) / (args.model if not args.repo else args.repo.split("/")[-1])
    ok = True
    for part in args.parts.split(","):
        if part == "verify":
            run_verify(model, bundle, folder / "verify", args.limit)
        elif part == "position":
            ok = run_position(model, bundle, folder / "position", conditions, args.sources) and ok
        else:
            raise ValueError(f"unknown part {part!r}")
    return 0 if ok else 2


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
os.chdir(ROOT)
assert os.path.exists(SCRIPT), "run the cell above first: it writes the script"
must(f"python {SCRIPT} selftest")                       # the script's own logic on tiny models, CPU and GPU paths
shutil.copyfile(f"{MAIN_RUN}/prepared.json", PREPARED)
print("prepared.json sha256:", hashlib.sha256(open(PREPARED, "rb").read()).hexdigest())
must(f"python {SCRIPT} preflight --prepared {PREPARED} --model {','.join(MODELS)}")   # tokenizers only
probe = f"{BACKUP}/.write-test"
with open(probe, "w") as f:
    f.write("ok")
assert open(probe).read() == "ok", "the Drive folder did not read back what was written"
os.remove(probe)
print("restored from Drive:", restore(), "files")
for model in MODELS:
    for part in PARTS:
        print(f"  {model:6s} {part:9s} {saved(model, part):5d} rows already saved")
note("ready: nothing below waits on you")


In [ ]:
os.chdir(ROOT)
status = {}
stop = threading.Event()
threading.Thread(target=watch, args=(stop,), daemon=True).start()
try:
    for model in MODELS:
        for part in PARTS:
            cmd = (f"python {SCRIPT} run --prepared {PREPARED} --model {model} --out {OUT} --parts {part} "
                   f"--conditions {CONDITIONS}")
            print(f"\n===== {model} {part} =====", flush=True)
            code = sh(cmd)
            if code != 0:                                 # one more try; it continues from the saved rows
                print(f"\n===== {model} {part}, second try =====", flush=True)
                code = sh(cmd)
            status[f"{model} {part}"] = code
            try:
                backup()
                note(f"{model} {part}: {'finished' if code == 0 else 'NOT finished (exit code ' + str(code) + ')'}, "
                     f"{saved(model, part)} rows in Drive")
            except Exception as error:
                note(f"{model} {part}: exit code {code}, but the copy to Drive failed ({type(error).__name__}: {error})")
finally:
    stop.set()
print(status)


In [ ]:
for model in MODELS:
    path = f"{OUT}/{model}/verify/summary.json"
    if os.path.exists(path):
        print(f"\n===== verify summary, {model} =====")
        print(json.dumps(json.load(open(path))["all"], indent=1))
try:
    backup()
except Exception as error:
    note(f"last copy to Drive failed ({type(error).__name__}: {error})")
note("done: " + ", ".join(f"{name} {'finished' if code == 0 else 'NOT finished'}" for name, code in status.items()))
try:
    drive.flush_and_unmount()
    print("Drive is up to date. Safe to disconnect and delete the runtime.")
except Exception as error:
    print("could not confirm the Drive sync:", type(error).__name__, error)
